# ETL Inicial
Importando a fonte de dados para o projeto e trazendo apenas as sheets do excel que iremos utilizar.

Feita uma limpeza nos dados que vinham do excel e preparado dois DataFrames para posterior análise

- `df` são os registros peça a peça
- `df_faturamento` são os registros finais de faturamento de cada mês
- `df_pedidos_pazello` são os registros dos pedidos feitos ao Pazello (Revisar)


In [85]:
# =========================================================================================================================================================================
# Tratamento do df
# =========================================================================================================================================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

ARQUIVO_EXCEL = 'Planilha de produção 2025.xlsx'

dict_excel =  pd.read_excel(ARQUIVO_EXCEL, sheet_name=None, header=2)

sheets_to_remove = ['Fechamento','Planilha2']

processed_dfs = []

for sheet_name, df_sheet in dict_excel.items():
    if sheet_name not in sheets_to_remove:
        df_temp = df_sheet.copy()
        df_temp['Mes'] = sheet_name
        processed_dfs.append(df_temp)


df = pd.concat(processed_dfs, ignore_index=True)

# Colunas Removidas
df = df.drop(columns=['%','Cuba','Unnamed: 16','Medição'])

# Renomeando Colunas
df.columns = df.columns.str.strip()

df = df.rename(columns={
    'Contrato': 'Nro Contrato',
    'Cliente': 'Cliente',
    'Itens': 'Itens',
    'Unnamed: 3': 'Tamanho', # Validar 
    'Material': 'Material',
    'M² orçado': 'M2 Orçado',
    'M²utilizado': 'M2 Utilizado',
    'Valor m²': 'M2 Valor',
    'L/P': 'Lucro Perda', # Validar
    'Material usado': 'Material Utilizado', # Colocar unidade
    'Total Produção': 'Total Produção (R$)',
    'Status': 'Status',
    'Unnamed: 14': 'Total Orçado (R$)', #Validar, não tenho a minha ideia do que é essa coluna
    'Colaborador': 'Colaborador',
    'Unnamed: 17': 'Total ?? (R$)' #Validar, essa tmb n sei
})


df = df.dropna(subset=['Nro Contrato'])

# Colunas Novas
df['Orçado x Utilizado (%)'] = np.where(
    (df['M2 Orçado'] == 0) | (df['M2 Utilizado'] == 0),
    0,
    ((df['M2 Utilizado'] * 100) / df['M2 Orçado'])
).round(2)

df['Tipo'] = np.where(
    df['Tamanho'].notna() & (df['Tamanho'] != ''),
    'Peça',
    'Outro'
)

# Tratamento das Strings
df['Cliente'] = df['Cliente'].astype(str).str.strip().str.title()
df['Itens'] = df['Itens'].astype(str).str.strip().str.title()
df['Tamanho'] = df['Tamanho'].astype(str).str.strip().str.title()
df['Status'] = df['Status'].astype(str).str.strip().str.title()
df['Material'] = df['Material'].astype(str).str.strip().str.title()
df['Colaborador'] = df['Colaborador'].astype(str).str.strip().str.title()
df['Mes'] = df['Mes'].astype(str).str.strip().str.title()

# Tratamento de tipos
df['Nro Contrato'] = df['Nro Contrato'].astype(int)


df.head(90)

,Nro Contrato,Cliente,Itens,Tamanho,Material,M2 Orçado,M2 Utilizado,M2 Valor,Lucro Perda,Material Utilizado,Total Produção (R$),Status,Total Orçado (R$),Colaborador,Total ?? (R$),Mes,Orçado x Utilizado (%),Tipo
0,6071,Guto E Sandra,Lavabo Diretoria,G,Psg Escovado,1.35,1.35,270.0,0.00,364.50,1754.0,Pronto,1754.0,Wellington,1299.259259,Janeiro,100.00,Peça
1,6071,Guto E Sandra,Copa,G,Psg,2.48,2.54,224.0,-13.44,568.96,3066.0,Pronto,3066.0,Pedro,1207.086614,Janeiro,102.42,Peça
2,6071,Guto E Sandra,Lavabo,G,Psg Escovado,0.79,0.79,270.0,0.00,213.30,1027.0,Pronto,2054.0,Wellington,2600.0,Janeiro,100.00,Peça
3,6071,Guto E Sandra,Lavabo,G,Psg Escovado,0.79,0.79,270.0,0.00,213.30,1027.0,Pronto,2054.0,Mateus,2600.0,Janeiro,100.00,Peça
4,6071,Guto E Sandra,Lavatório,G,Psg Escovado,2.14,2.14,270.0,0.00,577.80,2781.0,Pronto,2781.0,Wellington,1299.53271,Janeiro,100.00,Peça
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,6110,Bello Alimentos,Soleiras,G,Cinza Corumbá,0.54,0.54,125.0,0.00,67.50,200.0,Pronto,200.0,Wellington,370.37037,Março,100.00,Peça
123,6080,Jocelen,Bwc Hóspedes,M,Branco Moura,0.83,0.83,350.0,0.00,290.50,1527.0,Pronto,1527.0,Wellington,1839.759036,Março,100.00,Peça
124,6080,Jocelen,Bwc Quarto Tv,M,Branco Moura,0.62,0.62,350.0,0.00,217.00,1140.0,Pronto,1140.0,Wellington,1838.709677,Março,100.00,Peça
126,6054,Maria Quadri,Nicho Lareira,M,Premium,1.51,1.51,530.0,0.00,800.30,2574.0,Pronto,2574.0,Wellington,1704.635762,Março,100.00,Peça


In [88]:
# =========================================================================================================================================================================
# Tratamento do df_faturamento
# =========================================================================================================================================================================
df_fechamento =  pd.read_excel(ARQUIVO_EXCEL, sheet_name='Fechamento', header=4, usecols='A:K')

# Renomeando Colunas
df_fechamento.columns = df_fechamento.columns.str.strip()

df_fechamento = df_fechamento.rename(columns={
    'Unnamed: 0': 'Mes',
    'M2 orçado': 'M2 Orçado',
    'M2 utilizado': 'M2 Utilizado',
    'Material usado': 'Material Usado',
    'Cubas': 'Cubas',
    'Produção': 'Produção',
    'Custo Geral': 'Custo Geral',
    'Contas pagas': 'Contas Pagas',
    'Investimento': 'Investimento',
    'retrabalhos': 'Retrabalhos',
    'L/P material': 'Lucro Perda Material'
})

df_fechamento.head(50)

,Mes,M2 Orçado,M2 Utilizado,Material Usado,Cubas,Produção,Custo Geral,Contas Pagas,Investimento,Retrabalhos,Lucro Perda Material
0,Jan,110.24,105,59484.75,0,170466,80970.33,128395.59,1130.84,0,38.09
1,Fev,143.58,132.77,63871.05,0,208311,79925.28,124926.17,1130.84,0,53.9
2,Mar,122.93,122.37,76763.98,0,223579,85727.31,125015.3,1130.84,0,222.16
3,Abr,100.34,100.06,36823.5,0,144097,80533.95,163895.58,1130.84,1319,48.72
4,Mai,144.78,127.28,75211.73,1658,242109,87238.65,235605.52,1130.84,0,2193.68
5,Jun,118.38,117.99,53363.08,0,188149,83194.39,151670,1130.84,0,110.76
6,Jul,114.8,109.21,61837.42,192,194726,102893.18,179116.91,1130.84,0,335
7,Ago,124.19,91.51,54125.08,0,189515,87041.76,139321.43,1180.44,0,229.44
8,Set,141.99,134.81,81420.71,0,226072,100373.66,164531.51,1198.37,0,998.56
9,Out,199.66,185.05,87818.17,1096,300963,105508.35,167579.55,1189.41,0,1142.16


In [87]:
# =========================================================================================================================================================================
# Tratamento do df_pedido_pazello
# =========================================================================================================================================================================
df_pedido_pazello =  pd.read_excel(ARQUIVO_EXCEL, sheet_name='Fechamento', header=4, usecols='M:R')

df_pedido_pazello.head(50)

,Custo Prod. M2,Venda m2,INDICE RT,RT Acumulada,G,I
0,771.146000,1623.485714,9.41,74915.70,23.08,76.92
1,601.982978,1568.961362,6.83,61315.80,7.64,92.36
2,700.558225,1827.073629,8.36,64740.60,11.29,88.71
3,804.856586,1440.105936,9.14,66714.40,35.62,64.38
4,685.407370,1902.176304,6.97,78215.10,19.76,80.24
5,705.096957,1594.618188,0.83,61611.90,34.17,65.83
6,942.158960,1783.041846,3.82,53086.50,19.30,80.70
7,951.172112,2070.975850,5.84,58140.00,20.42,79.58
8,744.556487,1676.967584,5.70,54227.35,8.93,91.07
9,570.161308,1626.387463,6.06,67041.15,38.30,61.70
